# gpt-oss — the same model, two endpoints, and the reasoning hides in different places

`gpt-oss` is OpenAI's open-weight family. Unlike GPT-5.x, which is
`bedrock-mantle` only, gpt-oss is on **both** endpoints.

The models reason before answering, and **both endpoints return that reasoning —
but not in the same place, and one of them is not in any specification**:

| | `bedrock-runtime` | `bedrock-mantle` |
|---|---|---|
| Where the reasoning is | a `reasoningContent` block inside `content[]` | `choices[0].message.reasoning` |
| Is that documented? | yes, part of Converse | **no** — a non-standard extension to the OpenAI Chat Completions schema |
| How you find it | walk the typed content blocks | you have to know the field is there |

The `bedrock-mantle` case is the trap. Nothing in the OpenAI specification has a
`reasoning` field on a message, so SDK type hints and any code written against the
public schema will not surface it — you can be paying for reasoning and discarding
it without noticing. The same non-standard field carries Qwen's reasoning too, so
this is a `/v1`-family pattern rather than a gpt-oss quirk.

**The model ID also differs.** `openai.gpt-oss-120b` on `bedrock-mantle` is
`openai.gpt-oss-120b-1` on `bedrock-runtime` — note the `-1`. The safeguard
variants keep one ID on both. Sibling notebook 02 covers safeguard.


In [1]:
import sys

sys.path.insert(0, "../_shared")

from bedrock import (
    api_prefix,
    converse,
    converse_reasoning,
    endpoints_for,
    err,
    post,
    resolve_runtime_id,
)

REGION = "us-east-1"
MANTLE_120B = "openai.gpt-oss-120b"
RUNTIME_120B = "openai.gpt-oss-120b-1"
MANTLE_20B = "openai.gpt-oss-20b"
RUNTIME_20B = "openai.gpt-oss-20b-1"

print("mantle prefix for gpt-oss:", api_prefix(MANTLE_120B), "(not /openai/v1)")
print("endpoints:", endpoints_for(MANTLE_120B, REGION))
print()
for mantle_id, runtime_id in [(MANTLE_120B, RUNTIME_120B), (MANTLE_20B, RUNTIME_20B)]:
    print(f"  mantle {mantle_id:<24} -> runtime {resolve_runtime_id(runtime_id, REGION)}")


mantle prefix for gpt-oss: /v1 (not /openai/v1)


endpoints: {'mantle': True, 'runtime': True}



  mantle openai.gpt-oss-120b      -> runtime openai.gpt-oss-120b-1:0


  mantle openai.gpt-oss-20b       -> runtime openai.gpt-oss-20b-1:0


## 1. Find the reasoning on both endpoints

Same model, same question. The point of this cell is *where* the trace turns up,
not what it says.


In [2]:
QUESTION = "What is 17 * 23? Think it through, then give the answer."

# bedrock-runtime: a typed block among the content blocks.
text, response = converse(
    RUNTIME_120B,
    [{"role": "user", "content": [{"text": QUESTION}]}],
    max_tokens=600,
    region=REGION,
)
blocks = [next(iter(b)) for b in
          response.get("output", {}).get("message", {}).get("content", [])]
runtime_reasoning = converse_reasoning(response)
print("bedrock-runtime")
print("  content blocks :", blocks)
print("  reasoning found in: content[] -> reasoningContent")
print(f"  reasoning      : {len(runtime_reasoning)} chars")
print("  first 70       :", runtime_reasoning.strip()[:70])
print("  answer         :", text.strip()[:60])

# bedrock-mantle: NOT a block. A non-standard field on the message.
status, data = post(
    f"{api_prefix(MANTLE_120B)}/chat/completions",
    {
        "model": MANTLE_120B,
        "max_tokens": 600,
        "messages": [{"role": "user", "content": QUESTION}],
    },
    region=REGION,
)
print("\nbedrock-mantle")
if status != 200:
    print(f"  HTTP {status}: {err(data)}")
else:
    message = data["choices"][0]["message"]
    print("  message keys   :", sorted(message.keys()))
    print("  reasoning found in: message.reasoning  <- not in the OpenAI schema")
    mantle_reasoning = message.get("reasoning") or ""
    print(f"  reasoning      : {len(mantle_reasoning)} chars")
    print("  first 70       :", mantle_reasoning.strip()[:70])
    print("  answer         :", (message.get("content") or "").strip()[:60])
    print()
    print("  Code written against the published OpenAI schema reads .content and")
    print("  never looks at .reasoning - so it silently throws the trace away.")


bedrock-runtime
  content blocks : ['reasoningContent', 'text']
  reasoning found in: content[] -> reasoningContent
  reasoning      : 86 chars
  first 70       : We just compute 17*23. 17*20=340, 17*3=51, sum =391. So answer 391. Pr
  answer         : To multiply 17 by 23, break it into easier parts:

- \(17 \t



bedrock-mantle
  message keys   : ['content', 'reasoning', 'refusal', 'role']
  reasoning found in: message.reasoning  <- not in the OpenAI schema
  reasoning      : 119 chars
  first 70       : We just need to compute 17 * 23. 20*23=460, minus 3*23=69 gives 391. O
  answer         : To find \(17 \times 23\):

\[
\begin{aligned}
17 \times 23 &

  Code written against the published OpenAI schema reads .content and
  never looks at .reasoning - so it silently throws the trace away.


## 2. `reasoning_effort` changes the spend

More effort means more completion tokens. Worth measuring before you set it in
production, because the answer often does not change while the bill does.

Note what `completion_tokens_details` reports — on this endpoint the reasoning
tokens are **not** itemised, so you cannot separate thinking from answering in the
usage figures even though you can read the trace itself.


In [3]:
print(f"{'effort':<8} {'HTTP':<5} {'completion':>11} {'reasoning chars':>16}  answer")
print("-" * 78)
for effort in ("low", "medium", "high"):
    status, data = post(
        f"{api_prefix(MANTLE_120B)}/chat/completions",
        {
            "model": MANTLE_120B,
            "max_tokens": 700,
            "reasoning_effort": effort,
            "messages": [{"role": "user", "content": QUESTION}],
        },
        region=REGION,
    )
    if status != 200:
        print(f"{effort:<8} {status:<5} {'-':>11} {'-':>16}  {err(data)[:26]}")
        continue
    message = data["choices"][0]["message"]
    usage = data.get("usage", {})
    trace = message.get("reasoning") or ""
    answer = (message.get("content") or "").strip().replace("\n", " ")
    print(f"{effort:<8} {status:<5} {usage.get('completion_tokens'):>11} "
          f"{len(trace):>16}  {answer[:24]}")

print()
print("completion_tokens_details:", data.get("usage", {}).get("completion_tokens_details"))
print("=> None. Reasoning tokens are not itemised here, so budget from the")
print("   completion_tokens total rather than expecting a breakdown.")


effort   HTTP   completion  reasoning chars  answer
------------------------------------------------------------------------------


low      200            86               26  To find \(17 \times 23\)


medium   200           116               85  To multiply 17 by 23, br


high     200           246              518  To find \(17 \times 23\)

completion_tokens_details: None
=> None. Reasoning tokens are not itemised here, so budget from the
   completion_tokens total rather than expecting a breakdown.


## 3. 20b or 120b?

Two sizes, one API surface. Grade them on a task with a checkable answer rather
than comparing prose quality by eye.


In [4]:
PUZZLE = (
    "A shop sells pens at 3 for $4. How much for 18 pens? "
    "Reply with the number of dollars only."
)

print(f"{'model':<24} {'tokens':>7}  {'answer':<12} verdict")
print("-" * 62)
for runtime_id in (RUNTIME_20B, RUNTIME_120B):
    text, response = converse(
        runtime_id,
        [{"role": "user", "content": [{"text": PUZZLE}]}],
        max_tokens=600,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{runtime_id:<24} {'-':>7}  ERROR {error[:30]}")
        continue
    total = response.get("usage", {}).get("totalTokens", 0)
    answer = text.strip().replace("$", "").replace("\n", " ")
    verdict = "correct" if "24" in answer else "WRONG (expect 24)"
    print(f"{runtime_id:<24} {total:>7}  {answer[:12]:<12} {verdict}")


model                     tokens  answer       verdict
--------------------------------------------------------------


openai.gpt-oss-20b-1         178  24           correct


openai.gpt-oss-120b-1        160  24           correct


## Takeaways

- **gpt-oss is on both endpoints; GPT-5.x is not.** Within `01-openai-gpt/` and this
  folder the endpoint answer differs by model, even though the provider is the same.
- **The model ID gains a `-1` on `bedrock-runtime`** for the base models, and does not
  for the safeguard variants. There is no rule to infer — check per model.
- **Both endpoints return the reasoning, in different places.** Converse puts it in a
  typed `reasoningContent` block. `bedrock-mantle` puts it in
  `message.reasoning`, which is **not part of the OpenAI schema** — so code written
  against the published spec discards a trace you paid for. Qwen behaves the same
  way, so treat it as a `/v1`-family convention.
- **Reasoning tokens are not itemised** in `completion_tokens_details` on
  `bedrock-mantle`. Budget from the `completion_tokens` total.
- **gpt-oss lives on the `/v1` Mantle prefix**, not `/openai/v1` — the prefix follows
  the API family, not the vendor name.
